# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [27]:
#Libraries
import optuna
import optuna.visualization as vis
import time

import scipy.stats as st

from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, make_scorer

from sklearn.model_selection import cross_val_score

In [28]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [29]:
# Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


# -------------------------
# 1. DATA CLEANING
# -------------------------

# Drop missing values
spaceship = spaceship.dropna()

# Extract deck from Cabin
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]

# Remove unnecessary columns
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

# -------------------------
# 2. ENCODE CATEGORICAL VARIABLES
# -------------------------

spaceship_dummies = pd.get_dummies(spaceship, dtype=int)

# -------------------------
# 3. TRAIN TEST SPLIT
# -------------------------

features = spaceship_dummies.drop(columns=["Transported"])
target = spaceship_dummies["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=0
)

# -------------------------
# 4. FEATURE SCALING
# -------------------------

scaler = MinMaxScaler()

scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_test_norm = scaler.transform(X_test)

# Convert back to DataFrame
X_train_norm = pd.DataFrame(
    X_train_norm,
    columns=X_train.columns,
    index=X_train.index
)

X_test_norm = pd.DataFrame(
    X_test_norm,
    columns=X_test.columns,
    index=X_test.index
)

# -------------------------
# 5. FEATURE SELECTION
# -------------------------

# Correlation matrix
corr_matrix = X_train_norm.corr().abs()

# Upper triangle matrix
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find highly correlated features
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]

# Remove them
X_train_selected = X_train_norm.drop(columns=to_drop)
X_test_selected = X_test_norm.drop(columns=to_drop)

# -------------------------
# FINAL DATA
# -------------------------

print("Original feature count:", X_train.shape[1])
print("Remaining features after selection:", X_train_selected.shape[1])

Original feature count: 24
Remaining features after selection: 22


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [30]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=0
)

rf_model.fit(X_train_selected, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


- Evaluate your model

In [31]:
from sklearn.metrics import accuracy_score
y_pred_rf = rf_model.predict(X_test_selected)

rf_acc = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.7957639939485628


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [32]:
# First we need to setup a dictionary with all the values that we want to try for each hyperparameter

parameter_grid = {"max_depth": [10, 50],
                  "min_samples_split": [4, 16],
                  "max_leaf_nodes": [250, 100],
                  "max_features": ["sqrt", "log2"]}  # 2 * 2 * 2 * 2 = 16 combinations

# We create an instance of our machine learning model
dt = DecisionTreeRegressor(random_state=123)

# We need to set these two variables to compute a confidence interval
confidence_level = 0.95
folds = 10

- Run Grid Search

In [34]:
# Now we create an instance of the GridSearchCV class
gs = GridSearchCV(dt, param_grid=parameter_grid, cv=folds, verbose=10)

start_time = time.time()
gs.fit(X_train_selected, y_train)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")

print(f"The best combination of hyperparameters has been: {gs.best_params_}")
print(f"The R2 is: {gs.best_score_: .4f}")

results_gs_df = pd.DataFrame(gs.cv_results_).sort_values(by="mean_test_score", ascending=False)

Fitting 10 folds for each of 16 candidates, totalling 160 fits
[CV 1/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 1/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.209 total time=   0.0s
[CV 2/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 2/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.426 total time=   0.0s
[CV 3/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 3/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.344 total time=   0.0s
[CV 4/10; 1/16] START max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4
[CV 4/10; 1/16] END max_depth=10, max_features=sqrt, max_leaf_nodes=250, min_samples_split=4;, score=0.342 total time=   0.0s
[CV 5/10; 1/16] START max_depth=10, max_features=sqrt

- Evaluate your model

In [37]:
gs_mean_score = results_gs_df.iloc[0,-3]
gs_sem = results_gs_df.iloc[0,-2] / np.sqrt(folds)

gs_tc = st.t.ppf(1-((1-confidence_level)/2), df=folds-1)
gs_lower_bound = gs_mean_score - ( gs_tc * gs_sem )
gs_upper_bound = gs_mean_score + ( gs_tc * gs_sem )

print(f"The R2 confidence interval for the best combination of hyperparameters is: \
    ({gs_lower_bound: .4f}, {gs_mean_score: .4f}, {gs_upper_bound: .4f}) ")

# Let's store the best model
best_model = gs.best_estimator_

# Now it is time to evaluate the model in the test set
y_pred_test_df = best_model.predict(X_test_selected)

print("\n")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test_df): .4f}")
print(f"Test MSE: {mean_squared_error(y_test, y_pred_test_df): .4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, y_pred_test_df): .4f}")
print(f"Test R2 score:  {best_model.score(X_test_selected, y_test): .4f}")
print("\n")

The R2 confidence interval for the best combination of hyperparameters is:     ( 0.3289,  0.3610,  0.3930) 


Test MAE:  0.3024
Test MSE:  0.1636
Test RMSE:  0.4044
Test R2 score:   0.3457


